# SE446 Milestone 2 — Chicago Crime Analytics (Spark + MLlib)

**Group 5** — same repository as M1.

This notebook runs **Phase A (Tasks 1–4)** and **Phase B (Tasks 5–7)**. Set `RUN_MODE` in the next code cell (or `M2_RUN_MODE` env var):

- `local` — laptop, `local[*]`, 10k generated rows (Tasks 1–7; **Task 9**)
- `yarn_client` — cluster, `yarn`, `hdfs:///data/chicago_crimes.csv` (**Task 10**)

**Task 11:** use `m2_spark_ml.py` + `spark-submit`; save logs under `output/spark_submit/run.log`.

Replace `(ID: ________)` placeholders in code headers with real student IDs before submission.


In [ ]:
# ============================================
# Environment: local vs YARN (Tasks 9–10)
# Author: Alabalsaud (ID: ________)
# ============================================
import os

RUN_MODE = os.environ.get("M2_RUN_MODE", "local")

if RUN_MODE not in ("local", "yarn_client"):
    raise ValueError("M2_RUN_MODE must be 'local' or 'yarn_client'")

print("RUN_MODE =", RUN_MODE)


In [ ]:
from __future__ import annotations

import os
import time

import numpy as np
import pandas as pd

from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier, LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler


def build_spark_session(run_mode: str) -> SparkSession:
    b = SparkSession.builder.appName("SE446_M2_Group5")
    if run_mode == "local":
        b = b.master("local[*]")
    else:
        b = b.master("yarn").config("spark.ui.enabled", "false")

    spark = b.getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    print("Spark version:", spark.version)
    print("Master:", spark.sparkContext.master)
    return spark


spark = build_spark_session(RUN_MODE)


In [ ]:
# ============================================
# Synthetic Chicago sample (10k rows) — local only
# Author: Alabalsaud (ID: ________)
# ============================================
CRIME_TYPES = [
    "THEFT",
    "BATTERY",
    "CRIMINAL DAMAGE",
    "NARCOTICS",
    "ASSAULT",
    "MOTOR VEHICLE THEFT",
    "BURGLARY",
    "OTHER OFFENSE",
    "DECEPTIVE PRACTICE",
    "ROBBERY",
]
LOCATIONS = [
    "STREET",
    "RESIDENCE",
    "APARTMENT",
    "SIDEWALK",
    "PARKING LOT/GARAGE(NON.RESID.)",
    "OTHER",
    "GROCERY FOOD STORE",
    "DEPARTMENT STORE",
    "AIRPORT/AIRCRAFT",
    "BAR OR TAVERN",
]


def make_local_sample_df(spark: SparkSession, n: int = 10_000, seed: int = 42):
    rng = np.random.default_rng(seed)
    years = rng.integers(2001, 2026, size=n)
    districts = rng.integers(1, 26, size=n)
    hour = rng.integers(0, 24, size=n)
    domestic = rng.random(size=n) < 0.2

    ctype_idx = rng.integers(0, len(CRIME_TYPES), size=n)
    base_p = np.array([0.22, 0.25, 0.18, 0.55, 0.28, 0.15, 0.20, 0.12, 0.10, 0.30])
    p = base_p[ctype_idx] + 0.05 * domestic.astype(float)
    p = np.clip(p, 0.02, 0.85)
    arrest = rng.random(size=n) < p

    loc_idx = rng.integers(0, len(LOCATIONS), size=n)
    primary = np.array(CRIME_TYPES)[ctype_idx]
    loc = np.array(LOCATIONS)[loc_idx]

    month = rng.integers(1, 13, size=n)
    day = rng.integers(1, 29, size=n)
    minute = rng.integers(0, 60, size=n)
    second = rng.integers(0, 60, size=n)
    ampm = np.where(hour < 12, "AM", "PM")
    h12 = np.where((hour % 12) == 0, 12, hour % 12)
    date_str = [
        f"{m:02d}/{d:02d}/{y:04d} {hh:02d}:{mm:02d}:{ss:02d} {ap}"
        for m, d, y, hh, mm, ss, ap in zip(month, day, years, h12, minute, second, ampm)
    ]

    pdf = pd.DataFrame(
        {
            "Primary Type": primary,
            "Location Description": loc,
            "Arrest": arrest,
            "Domestic": domestic,
            "District": districts.astype(int),
            "Year": years.astype(int),
            "Date": date_str,
        }
    )
    return spark.createDataFrame(pdf)


if RUN_MODE == "local":
    df_raw = make_local_sample_df(spark, n=10_000, seed=42)
else:
    df_raw = (
        spark.read.option("header", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv("hdfs:///data/chicago_crimes.csv")
    )

print("Rows (raw read):", df_raw.count())
df_raw.printSchema()


In [ ]:
hour_ts1 = F.to_timestamp(F.col("Date"), "MM/dd/yyyy hh:mm:ss a")
hour_ts2 = F.to_timestamp(F.col("Date"), "MM/dd/yyyy HH:mm:ss")

df = (
    df_raw.select(
        F.col("Primary Type").alias("Primary Type"),
        F.col("Location Description").alias("Location Description"),
        F.col("Arrest").alias("Arrest"),
        F.col("Domestic").alias("Domestic"),
        F.col("District").cast("int").alias("District"),
        F.col("Year").cast("int").alias("Year"),
        F.col("Date").alias("Date"),
    )
    .withColumn(
        "label",
        F.when(F.lower(F.col("Arrest").cast("string")).isin("true", "1", "yes"), 1).otherwise(0),
    )
    .withColumn("Domestic_str", F.col("Domestic").cast("string"))
    .withColumn("Hour", F.coalesce(F.hour(hour_ts1), F.hour(hour_ts2)))
)

df = df.fillna({"District": -1, "Hour": -1}).filter(F.col("label").isin(0, 1))
df.cache()
print("Rows (usable):", df.count())


### Task 1: Crime type distribution (Spark DataFrame)
**Author: Halmineefi (ID: ________)**

**M1 reference (MapReduce, full dataset on cluster):** e.g. **THEFT (162,688)**, **BATTERY (151,930)**, **CRIMINAL DAMAGE (91,241)** — `output/task2/part-00000`.


In [ ]:
# ============================================
# Task 1: Crime Type Distribution
# Author: Halmineefi (ID: ________)
# ============================================
top_types = df.groupBy("Primary Type").count().orderBy(F.col("count").desc()).limit(10)
top_types.show(truncate=False)


### Task 2: Location hotspots (Spark SQL)
**Author: Mkalissa (ID: ________)**

**M1 reference:** **STREET (248,326)**, **RESIDENCE (136,393)**, … — `output/task3/part-00000`.


In [ ]:
# ============================================
# Task 2: Location Hotspots (must use spark.sql)
# Author: Mkalissa (ID: ________)
# ============================================
df.createOrReplaceTempView("crimes")

q = '''
SELECT `Location Description`, COUNT(*) AS total
FROM crimes
GROUP BY `Location Description`
ORDER BY total DESC
LIMIT 10
'''

spark.sql(q).show(truncate=False)


### Task 3: Crime trend over years + plot (local)
**Author: Lalmousa (ID: ________)**

**M1 reference:** year counts in `output/task4/part-00000`.


In [ ]:
# ============================================
# Task 3: Yearly trend
# Author: Lalmousa (ID: ________)
# ============================================
yearly = df.groupBy("Year").count().orderBy("Year")
yearly.show()

if RUN_MODE == "local":
    import matplotlib.pyplot as plt

    ypdf = yearly.toPandas()
    plt.figure(figsize=(9, 4))
    plt.plot(ypdf["Year"], ypdf["count"], marker="o")
    plt.title("Chicago crimes per year (local sample or full HDFS)")
    plt.xlabel("Year")
    plt.ylabel("Count")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("(Skipping matplotlib on yarn_client — table output above is sufficient.)")


### Task 4: Arrest rate — overall + by crime type (top 10)
**Author: Lalbabtain (ID: ________)**

**M1 overall:** Arrested **221,932** / total **793,072** → **~27.98%** (`output/task5/part-00000`).


In [ ]:
# ============================================
# Task 4: Arrest rate analysis
# Author: Lalbabtain (ID: ________)
# ============================================
overall = df.select(
    F.avg(F.col("label").cast("double")).alias("arrest_rate"),
    F.count(F.lit(1)).alias("n"),
)
overall.show(truncate=False)

by_type = df.groupBy("Primary Type").agg(
    F.avg(F.col("label").cast("double")).alias("arrest_rate"),
    F.count(F.lit(1)).alias("n"),
)

top10_by_volume = by_type.orderBy(F.col("n").desc()).limit(10)
print("Top 10 crime types by volume + arrest rate:")
top10_by_volume.show(truncate=False)

print("Among those 10: highest arrest rates")
top10_by_volume.orderBy(F.col("arrest_rate").desc()).show(10, truncate=False)

print("Among those 10: lowest arrest rates")
top10_by_volume.orderBy(F.col("arrest_rate").asc()).show(10, truncate=False)


### Task 5: Feature engineering + Pipeline
**Author: Alabalsaud (ID: ________)**

Assembled feature order: **District**, **crime_index**, **Hour**, **domestic_index**.


In [ ]:
# ============================================
# Task 5: Feature engineering
# Author: Alabalsaud (ID: ________)
# ============================================
seed = 42
train_df, test_df = df.randomSplit([0.8, 0.2], seed=seed)
train_df = train_df.persist(StorageLevel.MEMORY_AND_DISK)
test_df = test_df.persist(StorageLevel.MEMORY_AND_DISK)
train_df.count(), test_df.count()

crime_indexer = StringIndexer(
    inputCol="Primary Type",
    outputCol="crime_index",
    handleInvalid="skip",
)
domestic_indexer = StringIndexer(
    inputCol="Domestic_str",
    outputCol="domestic_index",
    handleInvalid="skip",
)
assembler = VectorAssembler(
    inputCols=["District", "crime_index", "Hour", "domestic_index"],
    outputCol="features",
    handleInvalid="skip",
)

prep = Pipeline(stages=[crime_indexer, domestic_indexer, assembler])
prep_model = prep.fit(train_df)
prep_model.transform(train_df).select("Primary Type", "Domestic", "District", "Hour", "features").limit(5).show(
    truncate=False
)


### Task 6: Train & compare three models
**Author: Alabalsaud (ID: ________)** *(split authorship with teammates as needed)*


In [ ]:
# ============================================
# Task 6: Metrics + training loop
# Author: Alabalsaud (ID: ________)
# ============================================
def confusion_counts(pred):
    tp = pred.filter((F.col("label") == 1) & (F.col("prediction") == 1)).count()
    tn = pred.filter((F.col("label") == 0) & (F.col("prediction") == 0)).count()
    fp = pred.filter((F.col("label") == 0) & (F.col("prediction") == 1)).count()
    fn = pred.filter((F.col("label") == 1) & (F.col("prediction") == 0)).count()
    return {"TN": tn, "FP": fp, "FN": fn, "TP": tp}


def eval_metrics(pred):
    auc = BinaryClassificationEvaluator(
        rawPredictionCol="rawPrediction",
        labelCol="label",
        metricName="areaUnderROC",
    ).evaluate(pred)

    acc = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="accuracy",
    ).evaluate(pred)
    f1 = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="f1",
    ).evaluate(pred)
    pr = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="weightedPrecision",
    ).evaluate(pred)
    rc = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="weightedRecall",
    ).evaluate(pred)

    return {
        "AUC-ROC": float(auc),
        "Accuracy": float(acc),
        "F1": float(f1),
        "Precision": float(pr),
        "Recall": float(rc),
    }


def build_classifier_pipeline(clf):
    return Pipeline(
        stages=[
            StringIndexer(
                inputCol="Primary Type",
                outputCol="crime_index",
                handleInvalid="skip",
            ),
            StringIndexer(
                inputCol="Domestic_str",
                outputCol="domestic_index",
                handleInvalid="skip",
            ),
            VectorAssembler(
                inputCols=["District", "crime_index", "Hour", "domestic_index"],
                outputCol="features",
                handleInvalid="skip",
            ),
            clf,
        ]
    )


models = {
    "LogisticRegression": LogisticRegression(
        featuresCol="features",
        labelCol="label",
        maxIter=100,
        regParam=0.01,
    ),
    "RandomForest": RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        numTrees=100,
        maxDepth=5,
        seed=seed,
    ),
    "GBT": GBTClassifier(
        featuresCol="features",
        labelCol="label",
        maxIter=50,
        maxDepth=5,
        seed=seed,
    ),
}

rows = []
fitted_rf = None

for name, clf in models.items():
    pipeline = build_classifier_pipeline(clf)
    t0 = time.perf_counter()
    model = pipeline.fit(train_df)
    train_s = time.perf_counter() - t0

    pred = model.transform(test_df)
    metrics = eval_metrics(pred)
    cm = confusion_counts(pred)

    rows.append({"Model": name, **metrics, **cm, "TrainSeconds": train_s})

    print("=" * 80)
    print(name, "train_s=", round(train_s, 3))
    print("metrics:", metrics)
    print("confusion:", cm)

    if name == "RandomForest":
        fitted_rf = model

summary = pd.DataFrame(rows).set_index("Model")
summary


### Task 7: Feature importances + interpretation
**Author: Lalbabtain (ID: ________)**


In [ ]:
# ============================================
# Task 7: Random Forest importances
# Author: Lalbabtain (ID: ________)
# ============================================
feats = ["District", "crime_index", "Hour", "domestic_index"]
rf_stage = fitted_rf.stages[-1]
imp = rf_stage.featureImportances.toArray()
ranked = sorted(zip(feats, imp), key=lambda x: x[1], reverse=True)
for f, v in ranked:
    print(f"{f:16s}  {float(v):.6f}")

if RUN_MODE == "local":
    import matplotlib.pyplot as plt

    xs = [x[0] for x in ranked]
    ys = [float(x[1]) for x in ranked]
    plt.figure(figsize=(7, 3))
    plt.bar(xs, ys)
    plt.title("Random Forest feature importances")
    plt.ylabel("Importance")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()


## Phase C — evidence checklist

- **Task 9:** `M2_RUN_MODE=local`, screenshot `Master: local[*]`.
- **Task 10:** SSH + `export M2_RUN_MODE=yarn_client`, screenshot `Master: yarn` + count **≥ 7,000,000** rows.
- **Task 11:** paste full `spark-submit` transcript and `yarn logs` excerpt into `output/spark_submit/run.log`.

**Interpretation prompts:** relate RF importances (especially `crime_index`) to per-type arrest rates in Task 4; explain LR vs trees (non-linearities + effect encoding).
